# Data Preparation Notebook

* #### This notebook prepares the turbulent jet LES [1] dataset for NO+DM training

* #### First, download the dataset from University of Michigan's Deep Blue Data Repository. 
    * Go to link (https://deepblue.lib.umich.edu/data/collections/kk91fk98z?locale=en).
    * Check section: Turbulent jet large eddy simulation
    * Download jet_3D_01000.zip



* #### If this link is not working, refer [1] 

* #### [1] - Towne, Aaron, et al. "A database for reduced-complexity modeling of fluid flows." AIAA journal 61.7 (2023): 2867-2892.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import matplotlib.pyplot as plt

from tqdm import tqdm

import matplotlib
matplotlib.rcParams["figure.dpi"] = 200
plt.rcParams["font.family"] = "serif"
import scipy.stats as stats

import pyvista as pv

In [2]:
# import zipfile
# import os

In [3]:
# zip_path = 'jet_3D_01000.zip'

# # Specify the directory to extract to
# extract_to_path = 'temp/'

# # Create the directory if it doesn't exist
# os.makedirs(extract_to_path, exist_ok=True)

# # Open the zip file
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     # Extract all the contents into the directory
#     zip_ref.extractall(extract_to_path)

# print("Extraction Complete!")


In [45]:
file = h5py.File(f'jet_grid.h5', 'r')
list(file)

x_grid = file["x"][:]
r_grid = file["r"][:]
theta_grid = file["theta"][:]

theta, r, x = np.meshgrid(theta_grid, r_grid, x_grid, indexing='ij')

# print(f"x_grid: {x_grid.shape}\n", x_grid)
# print(f"r_grid: {r_grid.shape}\n", r_grid)
# print(f"theta_grid: {theta_grid.shape}\n", theta_grid)
x.shape

(128, 138, 656)

# Saving the 3d velocity vectors

In [53]:
ux_ls = []
ur_ls = []
uth_ls = []
for i in tqdm(range(1,1001)):
    t_idx = str(100000+i)[1:]
    file = h5py.File(f'temp/jet_3D_t{t_idx}.h5', 'r')
    ux = file["ux"][::2, :64, :256] #nth, nr, nx
    ur = file["ur"][::2, :64, :256]
    uth = file["uth"][::2, :64, :256]

    ux_ls.append(ux)
    ur_ls.append(ur)
    uth_ls.append(uth)

ux = np.array(ux_ls)
ur = np.array(ur_ls)
uth = np.array(uth_ls)

velocity = np.array([uth, ur, ux]).transpose(1,0,2,3,4)

print(f"ux: {ux.shape}, {ux.dtype}")
print(f"ur: {ux.shape}, {ur.dtype}")
print(f"uth: {uth.shape}, {uth.dtype}")
print(f"velocity: {velocity.shape}, {velocity.dtype}")

np.save("velocity_vec_3d.npy", velocity)

100%|██████████| 1000/1000 [15:35<00:00,  1.07it/s]


ux: (1000, 64, 64, 256), float32
ur: (1000, 64, 64, 256), float32
uth: (1000, 64, 64, 256), float32
velocity: (1000, 3, 64, 64, 256), float32


In [60]:
velocity = np.load("velocity_vec_3d.npy")
velocity.shape, velocity.dtype

((1000, 3, 64, 64, 256), dtype('float32'))

In [61]:
MEAN = np.mean(velocity, axis=(0,2,3,4))
STD  = np.std(velocity, axis=(0,2,3,4))
MIN  = np.min(velocity, axis=(0,2,3,4))
MAX  = np.max(velocity, axis=(0,2,3,4))

np.save("MEAN_vel.npy", MEAN)
np.save("STD_vel.npy", STD)
np.save("MIN_vel.npy", MIN)
np.save("MAX_vel.npy", MAX)

In [55]:
velocity_sample = velocity[0:10] #[nt=10, nfields=3, nth=64, nr=64, nx=256]

file = h5py.File(f'jet_grid.h5', 'r')

theta_grid = file["theta"][::2]
r_grid = file["r"][:64]
x_grid = file["x"][:256]

np.save("visualization_sample/velocity.npy", velocity_sample)
np.save("visualization_sample/theta_grid.npy", theta_grid)
np.save("visualization_sample/r_grid.npy", r_grid)
np.save("visualization_sample/x_grid.npy", x_grid)


In [59]:
np.save("velocity_sample.npy", velocity[0:200])